# Assignment 1 (Week 2): Sensor OOP hierarchy
# تمرین ۱ (هفته‌ی ۲): سلسله‌مراتب شیءگرای حسگرها

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 0 &nbsp;|&nbsp;
**Released:** Week 2 &nbsp;|&nbsp; **Due:** start of Week 3 &nbsp;|&nbsp; **Effort:** about 4 hours

> **Provenance:** new content. OOP concepts follow (rewritten, not copied)
> [W3Schools Python Inheritance](https://www.w3schools.com/python/python_inheritance.asp),
> [W3Schools Polymorphism](https://www.w3schools.com/python/python_polymorphism.asp),
> [GeeksforGeeks Abstract Classes in Python](https://www.geeksforgeeks.org/python/abstract-classes-in-python/) and
> [GeeksforGeeks Getting Started with Pytest](https://www.geeksforgeeks.org/python/getting-started-with-pytest/).
> Flat-Earth local coordinates are adapted from Groves, *Principles of GNSS, Inertial, and
> Multisensor Integrated Navigation Systems*, Ch. 2.

<div dir="rtl">

## صورت مسئله

ربات زمینی آزمایشی ما در فرودگاه مهرآباد تهران یک دور دایره‌ای به شعاع ۴۰ متر می‌زند و سپس در یک سالن ۱۰×۶ متری که یک ستون در آن قرار دارد حرکت می‌کند. دو حسگر جدید روی ربات نصب شده است:

* **GPS** با نرخ ۵ هرتز که عرض جغرافیایی، طول جغرافیایی، ارتفاع و HDOP (معیار کیفیت هندسه‌ی ماهواره‌ها) را گزارش می‌کند. برخی موقعیت‌ها خراب‌اند: HDOP بزرگ، مقدار NaN و یک عرض جغرافیایی غیرممکن.
* **LiDAR** دوبعدی با ۱۶ پرتو و نرخ ۱۰ هرتز. پرتوهایی که بازتاب ندارند مقدار ۰٫۰ یا بی‌نهایت می‌دهند، گردوغبار بازتاب‌های خیلی نزدیک (۵ سانتی‌متر) ایجاد می‌کند و چند اسکن به‌خاطر مه تقریباً کور هستند.

وظیفه‌ی شما نوشتن دو کلاس <code>GPSSensor</code> و <code>LidarSensor</code> است که هر دو از کلاس پایه‌ی انتزاعی <code>BaseSensor</code> (کتابخانه‌ی درس، <code>src/aerospace_ai/sensors.py</code>) ارث می‌برند و فقط دو متد <code>validate</code> و <code>preprocess</code> و چند ویژگی خاص خود را پیاده می‌کنند. سپس یک تابع چندریختی (polymorphic) گزارش می‌نویسید و برای کد خود آزمون‌های <code>pytest</code> طراحی می‌کنید.

**قواعد:**
- کد را در فایل <code>sensor_hierarchy.py</code> و آزمون‌ها را در <code>test_sensor_hierarchy.py</code> بنویسید؛ این نوت‌بوک فقط کد شما را بارگذاری و بررسی می‌کند.
- فقط بلوک‌هایی را که با <code># TODO</code> مشخص شده‌اند تغییر دهید. نام کلاس‌ها و متدها را تغییر ندهید.
- سلول‌های «HIDDEN TESTS» را پاک یا ویرایش نکنید؛ مدرس هنگام نمره‌دهی آن‌ها را جایگزین می‌کند.
- همه‌ی توضیحات داخل کد به زبان انگلیسی باشد.

</div>

**English summary.** Implement `GPSSensor` and `LidarSensor` as subclasses of the course's
abstract `BaseSensor`, write a polymorphic `sensor_report`, and write your own pytest suite.
Edit `sensor_hierarchy.py` and `test_sensor_hierarchy.py` in this folder; this notebook
loads, checks and visualises your code. Do not delete the `HIDDEN TESTS` cells.

| Task | Points | Where |
|------|-------:|-------|
| 1. `GPSSensor.__init__` and `origin` | 10 | `sensor_hierarchy.py` |
| 2. `GPSSensor.validate` | 15 | `sensor_hierarchy.py` |
| 3. `GPSSensor.preprocess`, `distance_travelled_m` | 15 | `sensor_hierarchy.py` |
| 4. `LidarSensor.channel_names`, `angles_deg` | 10 | `sensor_hierarchy.py` |
| 5. `LidarSensor._in_range`, `validate`, `preprocess` | 15 | `sensor_hierarchy.py` |
| 6. `LidarSensor.nearest_obstacle` | 10 | `sensor_hierarchy.py` |
| 7. `sensor_report` (polymorphism) | 10 | `sensor_hierarchy.py` |
| 8. Your pytest suite | 15 | `test_sensor_hierarchy.py` |

## Setup (run first, do not edit)

In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path
from types import ModuleType
from typing import Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
ASSIGNMENT_DIR: Final[Path] = REPO_ROOT / "assignments" / "assignment01"
for folder in (REPO_ROOT / "src", ASSIGNMENT_DIR):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

GPS_CSV: Final[Path] = REPO_ROOT / "data" / "sample" / "assignment01_gps_track.csv"
LIDAR_CSV: Final[Path] = REPO_ROOT / "data" / "sample" / "assignment01_lidar_scans.csv"
GPS_RAW_COLUMNS: Final[list[str]] = ["lat_deg", "lon_deg", "alt_m", "hdop"]
ORIGIN_LAT_DEG: Final[float] = 35.6892    # Tehran Mehrabad airport
ORIGIN_LON_DEG: Final[float] = 51.3134
ORIGIN_ALT_M: Final[float] = 1190.0
GPS_RATE_HZ: Final[float] = 5.0
LIDAR_RATE_HZ: Final[float] = 10.0


def load_student_module() -> ModuleType:
    """Import (or re-import after an edit) the student's sensor_hierarchy.py."""
    module = importlib.import_module("sensor_hierarchy")
    return importlib.reload(module)


def status(task: str, func) -> None:
    """Run a public check; print [ ] while the task still raises NotImplementedError."""
    try:
        func()
    except NotImplementedError:
        print(f"[ ] {task}: not done yet")
        return
    print(f"[x] {task}: public checks passed")


sh = load_student_module()
gps_log = pd.read_csv(GPS_CSV)
lidar_log = pd.read_csv(LIDAR_CSV)
print("GPS log:", gps_log.shape, "| LiDAR log:", lidar_log.shape)
gps_log.head(3)

### Background: what you inherit

`BaseSensor` already provides: `sensor_id`, `rate_hz` (validated), `period_s`,
`channel_names` (returns the class attribute `CHANNELS`), `n_channels`, `n_rejected`,
`latest`, `add_reading` (shape check → time check → `validate` → `preprocess` → store),
`add_readings`, `readings(valid_only)`, `timestamps()`, `to_array()`, `to_frame()`,
`__len__`, `__iter__`, `__repr__`. Read `src/aerospace_ai/sensors.py`, and use
`IMUSensor` and `BarometerSensor` there as models.

**Flat-Earth local frame (by hand).** With Earth radius $R = 6\,371\,000$ m one degree of
latitude is $R\pi/180 = 111\,195$ m everywhere. One degree of longitude shrinks with
latitude: at Mehrabad ($\varphi_0 = 35.6892°$, $\cos\varphi_0 = 0.81221$) it is
$111\,195 \times 0.81221 = 90\,314$ m. So a fix 0.001° north and 0.001° east of the origin
is at (east, north) = (90.3 m, 111.2 m). The approximation error grows with distance squared;
it is below 1 cm over our 80 m track.

### Task 1 (10 pts): `GPSSensor.__init__` and `origin`
<div dir="rtl">سازنده‌ی کلاس GPS را کامل کنید: ابتدا با <code>super()</code> سازنده‌ی کلاس پایه را صدا بزنید، سپس مبدأ و <code>max_hdop</code> را بررسی و ذخیره کنید. ویژگی <code>origin</code> باید فقط‌خواندنی باشد.</div>

Call the base-class constructor with `super()`, validate the origin and `max_hdop`
(raise `ValueError`), store the origin as a tuple of three floats, and expose it through a
read-only property. `LidarSensor.__init__` (given) is your model.

In [ ]:
def _check_task1() -> None:
    sh = load_student_module()
    gps = sh.GPSSensor("gps0", GPS_RATE_HZ, ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M)
    assert gps.origin == (ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M)
    assert gps.rate_hz == GPS_RATE_HZ and len(gps) == 0
    print(gps)


status("Task 1", _check_task1)

In [ ]:
# HIDDEN TESTS (task1): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 2 (15 pts): `GPSSensor.validate`
<div dir="rtl">یک موقعیت فقط وقتی معتبر است که مختصات معتبر باشد، ارتفاع و HDOP متناهی باشند و 0 &lt; HDOP ≤ max_hdop.</div>

A fix is valid only if the coordinate is valid, altitude and HDOP are finite and
$0 < \text{HDOP} \le$ `max_hdop`. Return a plain `bool`.

In [ ]:
def _check_task2() -> None:
    sh = load_student_module()
    gps = sh.GPSSensor("gps0", GPS_RATE_HZ, ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M)
    assert gps.validate(np.array([ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M, 1.0]))
    assert not gps.validate(np.array([ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M, 8.0]))
    n_valid = gps.add_readings(gps_log["timestamp_s"], gps_log[GPS_RAW_COLUMNS].to_numpy())
    print(f"valid fixes: {n_valid}, rejected: {gps.n_rejected}")


status("Task 2", _check_task2)

In [ ]:
# HIDDEN TESTS (task2): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 3 (15 pts): `GPSSensor.preprocess` and `distance_travelled_m`
<div dir="rtl">مختصات جغرافیایی را با تقریب زمین تخت به متر (شرق، شمال، بالا) تبدیل کنید و طول مسیر افقی را بدون حلقه‌ی پایتون حساب کنید.</div>

Convert to (east, north, up, hdop) in metres with the formulas in the class docstring, and
compute the horizontal path length through all valid fixes with `np.diff` and
`np.linalg.norm` (no Python loop).

In [ ]:
def _check_task3() -> None:
    sh = load_student_module()
    gps = sh.GPSSensor("gps0", GPS_RATE_HZ, ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M)
    east, north, up, hdop = gps.preprocess(
        np.array([ORIGIN_LAT_DEG + 0.001, ORIGIN_LON_DEG + 0.001, ORIGIN_ALT_M + 2.0, 0.9])
    )
    print(f"0.001 deg NE of the origin -> east {east:.2f} m, north {north:.2f} m, up {up:.2f} m")
    assert np.isclose(east, 90.31, atol=0.01) and np.isclose(north, 111.19, atol=0.01)
    gps.add_readings(gps_log["timestamp_s"], gps_log[GPS_RAW_COLUMNS].to_numpy())
    print(f"distance travelled: {gps.distance_travelled_m():.1f} m "
          f"(an ideal 40 m circle is {2 * np.pi * 40:.1f} m; why is ours longer?)")


status("Task 3", _check_task3)

In [ ]:
# HIDDEN TESTS (task3): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 4 (10 pts): `LidarSensor.channel_names` and `angles_deg`
<div dir="rtl">LiDAR تعداد کانال ثابتی ندارد؛ پس به‌جای ثابت کلاسی <code>CHANNELS</code>، ویژگی <code>channel_names</code> را بازنویسی (override) کنید. زاویه‌ی پرتوها را با یک عبارت NumPy بسازید.</div>

A LiDAR's number of channels depends on `n_beams`, so instead of a class-level `CHANNELS` it
*overrides* the `channel_names` property. Build `angles_deg` with one NumPy expression.

In [ ]:
def _check_task4() -> None:
    sh = load_student_module()
    lidar = sh.LidarSensor("lidar0", LIDAR_RATE_HZ)
    print(lidar.channel_names[:3], "...", lidar.n_channels, "channels")
    print("angles:", lidar.angles_deg[:4], "...")
    assert lidar.n_channels == 16 and lidar.angles_deg[8] == 0.0


status("Task 4", _check_task4)

In [ ]:
# HIDDEN TESTS (task4): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 5 (15 pts): `LidarSensor._in_range`, `validate`, `preprocess`
<div dir="rtl">پرتویی قابل استفاده است که برد آن متناهی و بین حداقل و حداکثر برد باشد. اسکن وقتی معتبر است که دست‌کم کسر <code>min_valid_fraction</code> از پرتوها قابل استفاده باشند. در پیش‌پردازش، پرتوهای غیرقابل استفاده NaN می‌شوند.</div>

A beam is usable when its range is finite and inside `[min_range_m, max_range_m]`. A scan is
valid when at least `min_valid_fraction` of its beams are usable. Preprocessing replaces
unusable beams by NaN. Your code must not emit NumPy warnings on `inf` or NaN inputs.

In [ ]:
def _check_task5() -> None:
    sh = load_student_module()
    lidar = sh.LidarSensor("lidar0", LIDAR_RATE_HZ, n_beams=4)
    print(lidar.preprocess(np.array([2.0, 0.0, np.inf, 0.05])))
    lidar = sh.LidarSensor("lidar0", LIDAR_RATE_HZ)
    n_valid = lidar.add_readings(lidar_log["timestamp_s"], lidar_log.iloc[:, 1:].to_numpy())
    print(f"valid scans: {n_valid}, rejected: {lidar.n_rejected}")


status("Task 5", _check_task5)

In [ ]:
# HIDDEN TESTS (task5): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 6 (10 pts): `LidarSensor.nearest_obstacle`
<div dir="rtl">فاصله و زاویه‌ی نزدیک‌ترین مانع را در آخرین اسکن معتبر برگردانید و پرتوهای NaN را نادیده بگیرید.</div>

Return `(distance_m, angle_deg)` of the closest echo in the **latest valid** scan, ignoring
NaN beams (`np.nanargmin`). Raise `ValueError` if there is no valid scan.

In [ ]:
def _check_task6() -> None:
    sh = load_student_module()
    lidar = sh.LidarSensor("lidar0", LIDAR_RATE_HZ)
    lidar.add_readings(lidar_log["timestamp_s"], lidar_log.iloc[:, 1:].to_numpy())
    distance, angle = lidar.nearest_obstacle()
    print(f"nearest obstacle: {distance:.3f} m at {angle:+.1f} deg")


status("Task 6", _check_task6)

In [ ]:
# HIDDEN TESTS (task6): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 7 (10 pts): `sensor_report`
<div dir="rtl">یک تابع بنویسید که برای هر فهرستی از حسگرها (حتی حسگری که هنوز ساخته نشده) جدول خلاصه بسازد. از <code>isinstance</code> برای نوع خاص حسگر استفاده نکنید.</div>

Build one summary table for **any** list of sensors, including types you have never seen
(the hidden test passes a `BarometerSensor`). Use only the `BaseSensor` interface; no
`isinstance` checks on specific sensor types.

In [ ]:
def _check_task7() -> None:
    sh = load_student_module()
    gps = sh.GPSSensor("gps0", GPS_RATE_HZ, ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M)
    gps.add_readings(gps_log["timestamp_s"], gps_log[GPS_RAW_COLUMNS].to_numpy())
    lidar = sh.LidarSensor("lidar0", LIDAR_RATE_HZ)
    lidar.add_readings(lidar_log["timestamp_s"], lidar_log.iloc[:, 1:].to_numpy())
    print(sh.sensor_report([gps, lidar]))


status("Task 7", _check_task7)

In [ ]:
# HIDDEN TESTS (task7): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 8 (15 pts): your pytest suite
<div dir="rtl">در فایل <code>test_sensor_hierarchy.py</code> دست‌کم ۸ تابع آزمون بنویسید (دو نمونه داده شده است) و از <code>pytest.raises</code>، <code>parametrize</code> و یک fixture استفاده کنید. مدرس آزمون‌های شما را روی سه پیاده‌سازی عمداً معیوب نیز اجرا می‌کند؛ هر پیاده‌سازی معیوبی که آزمون‌های شما رد کنند امتیاز دارد.</div>

Write at least 8 test functions in `test_sensor_hierarchy.py` (two examples are given), using
`pytest.raises`, `@pytest.mark.parametrize` and a fixture. Tests must be self-contained (no
CSV files). Scoring: 3 pts for ≥ 8 tests that all pass on a correct implementation, 3 pts for
using the three features, and 3 pts for each of three subtly broken implementations your
suite catches. **Hint:** check numbers, not just types.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings",
     "test_sensor_hierarchy.py"],
    cwd=ASSIGNMENT_DIR, capture_output=True, text=True,
)
print(result.stdout[-2000:])

In [ ]:
# HIDDEN TESTS (task8): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

## Visual check (not graded)

The GPS track should be one circle starting at (0, 0) and reaching about 80 m east; the
LiDAR points of the last valid scan should outline the hall walls and the pillar.

In [ ]:
try:
    sh = load_student_module()
    gps = sh.GPSSensor("gps0", GPS_RATE_HZ, ORIGIN_LAT_DEG, ORIGIN_LON_DEG, ORIGIN_ALT_M)
    gps.add_readings(gps_log["timestamp_s"], gps_log[GPS_RAW_COLUMNS].to_numpy())
    lidar = sh.LidarSensor("lidar0", LIDAR_RATE_HZ)
    lidar.add_readings(lidar_log["timestamp_s"], lidar_log.iloc[:, 1:].to_numpy())
    enu = gps.to_array()
    ranges = lidar.to_array()[-1]
    theta = np.radians(lidar.angles_deg)

    fig, (ax_gps, ax_lidar) = plt.subplots(1, 2, figsize=(11, 4.5))
    ax_gps.plot(enu[:, 0], enu[:, 1], ".", ms=3)
    ax_gps.set_xlabel("East [m]")
    ax_gps.set_ylabel("North [m]")
    ax_gps.set_title("GPS track (valid fixes)")
    ax_gps.set_aspect("equal")
    ax_lidar.plot(ranges * np.cos(theta), ranges * np.sin(theta), "o")
    ax_lidar.plot(0, 0, "k^", ms=10, label="robot (heading +x)")
    ax_lidar.set_xlabel("x forward [m]")
    ax_lidar.set_ylabel("y left [m]")
    ax_lidar.set_title("Last valid LiDAR scan")
    ax_lidar.set_aspect("equal")
    ax_lidar.legend(loc="lower left")
    for ax in (ax_gps, ax_lidar):
        ax.grid(alpha=0.3)
    fig.tight_layout()
except NotImplementedError:
    print("[ ] finish Tasks 1 to 6 to see the plots")

## Expected output (check yourself against these)

| Quantity | Expected value |
|----------|----------------|
| GPS log / LiDAR log shape | (300, 5) / (50, 17) |
| 0.001° N and 0.001° E of the origin | east 90.31 m, north 111.19 m |
| GPS fixes: valid / rejected | 288 / 12 (8 high HDOP, 3 NaN, 1 latitude of 135.69°) |
| Distance travelled | ≈ 284.0 m (ideal circle 251.3 m; position noise adds length) |
| Maximum east coordinate | ≈ 80 m |
| LiDAR angles | −180, −157.5, …, 157.5 deg; beam 8 points forward (0°) |
| LiDAR scans: valid / rejected | 46 / 4 (the fog-blinded scans) |
| Nearest obstacle, last valid scan | 1.875 m at −22.5° (the pillar, front-right) |
| `sensor_report` | gps0: 300 / 288 / 12 / 4.0 %; lidar0: 50 / 46 / 4 / 8.0 % |
| Your pytest suite | ≥ 8 tests, all passing |

## Submission checklist

* *Kernel → Restart & Run All* completes without errors.
* `python -m pytest assignments/assignment01 -q` passes.
* Type hints and Google-style docstrings on everything you wrote; English comments only;
  no magic numbers (use the module constants).
* Commit `sensor_hierarchy.py`, `test_sensor_hierarchy.py` and this executed notebook to your
  branch with at least three meaningful commits.